# Ingest Subscription Network and Competitive Universe

This notebook fetches a target channel's comment authors, retrieves their subscriptions, and identifies a "competitive universe" of similar channels. It then extracts top-performing videos from these similar channels based on semantic similarity to the base channel.

## Environment Setup and Authentication

Install dependencies and establish connections to Google Drive and YouTube API.

In [ ]:
# Install dependencies
!pip install -q google-api-python-client pinecone pandas numpy scikit-learn joblib tqdm

import os
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from tqdm.auto import tqdm
try:
    from google.colab import drive, userdata
except ImportError:
    drive, userdata = None, None
from googleapiclient.discovery import build
from pinecone import Pinecone
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# Mount Google Drive
try:
    if drive:
        drive.mount('/content/drive')
        print("✅ Drive mounted")
except Exception as e:
    print(f"⚠️ Drive mount failed (local execution?): {e}")

# Initialize YouTube API
try:
    YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY') if userdata else os.getenv('YOUTUBE_API_KEY')
    if YOUTUBE_API_KEY:
        youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
        print("✅ YouTube API initialized")
    else:
        print("⚠️ No YouTube API key found")
except Exception as e:
    print(f"❌ YouTube API setup failed: {e}")

# Initialize Pinecone
try:
    PINECONE_API_KEY = userdata.get('PINECONE_API_KEY') if userdata else os.getenv('PINECONE_API_KEY')
    if PINECONE_API_KEY:
        pc = Pinecone(api_key=PINECONE_API_KEY)
        pinecone_index = pc.Index('finder')
        print("✅ Pinecone connected")
    else:
        print("⚠️ No Pinecone API key found")
except Exception as e:
    print(f"❌ Pinecone setup failed: {e}")

## Configuration and Constants

In [ ]:
BASE_CHANNEL_ID = 'UC6t1O76G0jYXOAoYCm153dA' # Example: Lenny's Podcast
MAX_COMMENTS = 1000
MAX_AUTHORS = 500
MAX_SUBS_PER_AUTHOR = 1000
TOP_CHANNELS_COUNT = 1000
SIMILAR_CHANNELS_COUNT = 100
MIN_VIDEOS = 25
MAX_VIDEOS = 50

DRIVE_BASE_DIR = Path('/content/drive/MyDrive/Graphiko') if drive else Path('./Graphiko')
CACHE_DIR = DRIVE_BASE_DIR / 'cache'
COMMENTS_CACHE_DIR = CACHE_DIR / 'comments'
SUBS_CACHE_DIR = CACHE_DIR / 'subscriptions'
CHANNELS_CACHE_DIR = CACHE_DIR / 'channels'
VIDEOS_CACHE_DIR = CACHE_DIR / 'videos'

for d in [COMMENTS_CACHE_DIR, SUBS_CACHE_DIR, CHANNELS_CACHE_DIR, VIDEOS_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def get_channel_name(channel_id):
    if 'youtube' not in globals() or youtube is None: return channel_id
    try:
        res = youtube.channels().list(part='snippet', id=channel_id).execute()
        if res['items']:
            return res['items'][0]['snippet']['title']
    except:
        pass
    return channel_id

BASE_CHANNEL_NAME = get_channel_name(BASE_CHANNEL_ID)
print(f"Target Channel: {BASE_CHANNEL_NAME} ({BASE_CHANNEL_ID})")

## 1. Fetch Comments and Authors

We retrieve up to 1,000 comments from the target channel and extract up to 500 unique author IDs. Caching is used to avoid redundant API calls.

In [ ]:
def get_comment_authors(channel_id, max_comments=1000):
    cache_file = COMMENTS_CACHE_DIR / f"{channel_id}_authors.json"
    if cache_file.exists():
        with open(cache_file, 'r') as f:
            return json.load(f)

    if 'youtube' not in globals() or youtube is None: return []
    authors = set()
    next_page_token = None
    total_fetched = 0
    
    pbar = tqdm(total=max_comments, desc="Fetching comments")
    while total_fetched < max_comments:
        try:
            res = youtube.commentThreads().list(
                part='snippet',
                allThreadsAtChannelId=channel_id,
                maxResults=min(100, max_comments - total_fetched),
                pageToken=next_page_token
            ).execute()
            
            for item in res.get('items', []):
                author_id = item['snippet']['topLevelComment']['snippet'].get('authorChannelId', {}).get('value')
                if author_id:
                    authors.add(author_id)
            
            total_fetched += len(res.get('items', []))
            pbar.update(len(res.get('items', [])))
            next_page_token = res.get('nextPageToken')
            if not next_page_token: break
        except Exception as e:
            print(f"Error fetching comments: {e}")
            break
    pbar.close()
    
    author_list = list(authors)
    with open(cache_file, 'w') as f:
        json.dump(author_list, f)
    return author_list

all_authors = get_comment_authors(BASE_CHANNEL_ID, MAX_COMMENTS)
target_authors = all_authors[:MAX_AUTHORS]
print(f"Extracted {len(target_authors)} unique authors from {len(all_authors)} total authors found.")

## 2. Fetch Author Subscriptions (with Caching)

For each author, we retrieve their public subscriptions. We store these locally to avoid redundant API calls in future runs.

In [ ]:
def get_author_subscriptions(author_id, max_subs=1000):
    cache_file = SUBS_CACHE_DIR / f"{author_id}.json"
    if cache_file.exists():
        with open(cache_file, 'r') as f:
            return json.load(f)
    
    if 'youtube' not in globals() or youtube is None: return []
    
    subs = []
    next_page_token = None
    
    try:
        while len(subs) < max_subs:
            res = youtube.subscriptions().list(
                part='snippet',
                channelId=author_id,
                maxResults=min(50, max_subs - len(subs)),
                pageToken=next_page_token
            ).execute()
            
            for item in res.get('items', []):
                subs.append({
                    'id': item['snippet']['resourceId']['channelId'],
                    'title': item['snippet']['title']
                })
            
            next_page_token = res.get('nextPageToken')
            if not next_page_token: break
            
        with open(cache_file, 'w') as f:
            json.dump(subs, f)
        return subs
    except Exception as e:
        # Often subscriptions are private, which throws a 403
        # Cache empty list for errors to avoid repeated failing calls
        with open(cache_file, 'w') as f:
            json.dump([], f)
        return []

all_subscriptions = []
for author_id in tqdm(target_authors, desc="Fetching subscriptions"):
    subs = get_author_subscriptions(author_id, MAX_SUBS_PER_AUTHOR)
    all_subscriptions.extend(subs)

## 3. Aggregate Subscriptions

We count the frequency of each subscribed channel to identify which channels are most popular among the target channel's audience.

In [ ]:
df_subs = pd.DataFrame(all_subscriptions)
if not df_subs.empty:
    agg_subs = df_subs.groupby('id').agg({
        'title': 'first',
        'id': 'count'
    }).rename(columns={'id': 'subscriber_count'}).sort_values('subscriber_count', ascending=False)
    
    top_channels = agg_subs.head(TOP_CHANNELS_COUNT).reset_index()
    print(f"Found {len(top_channels)} candidate channels for the competitive universe.")
    print(top_channels.head(10))
else:
    top_channels = pd.DataFrame(columns=['id', 'title', 'subscriber_count'])
    print("No subscriptions found (public profiles might be rare).")

## 4. Fetch Channel Details and Embed Descriptions

We fetch descriptions for the top 1,000 channels and use them to calculate semantic similarity to the base channel.

In [ ]:
def fetch_channel_descriptions(channel_ids):
    descriptions = {}
    missing_ids = []
    
    for ch_id in channel_ids:
        cache_file = CHANNELS_CACHE_DIR / f"{ch_id}.json"
        if cache_file.exists():
            with open(cache_file, 'r') as f:
                descriptions[ch_id] = json.load(f).get('description', '')
        else:
            missing_ids.append(ch_id)
            
    if not missing_ids or 'youtube' not in globals() or youtube is None:
        return descriptions

    for i in range(0, len(missing_ids), 50):
        batch = missing_ids[i:i+50]
        try:
            res = youtube.channels().list(part='snippet', id=','.join(batch)).execute()
            for item in res.get('items', []):
                ch_id = item['id']
                desc = item['snippet'].get('description', '')
                descriptions[ch_id] = desc
                with open(CHANNELS_CACHE_DIR / f"{ch_id}.json", 'w') as f:
                    json.dump({'description': desc, 'title': item['snippet']['title']}, f)
        except Exception as e:
            print(f"Error fetching channel details: {e}")
            
    return descriptions

channel_ids = top_channels['id'].tolist()
if BASE_CHANNEL_ID not in channel_ids: channel_ids.append(BASE_CHANNEL_ID)

descriptions_map = fetch_channel_descriptions(channel_ids)
top_channels['description'] = top_channels['id'].map(descriptions_map).fillna('')
base_description = descriptions_map.get(BASE_CHANNEL_ID, "")

def embed_texts(texts):
    if 'pc' not in globals() or pc is None: return []
    if not texts: return []
    try:
        res = pc.inference.embed(
            model="multilingual-e5-large",
            inputs=texts,
            parameters={"input_type": "passage"}
        )
        return [emb['values'] for emb in res.data]
    except Exception as e:
        print(f"Embedding error: {e}")
        return [None] * len(texts)

# Embed base channel description
base_emb_list = embed_texts([base_description])
base_emb = base_emb_list[0] if base_emb_list else None

# Embed top channel descriptions in batches
channel_descriptions = top_channels['description'].tolist()
channel_embeddings = []
for i in tqdm(range(0, len(channel_descriptions), 96), desc="Embedding descriptions"):
    batch = channel_descriptions[i:i+96]
    channel_embeddings.extend(embed_texts(batch))

if base_emb is not None and any(e is not None for e in channel_embeddings):
    # Filter out None embeddings for similarity calculation
    valid_indices = [i for i, e in enumerate(channel_embeddings) if e is not None]
    valid_embs = [channel_embeddings[i] for i in valid_indices]
    
    similarities = np.zeros(len(channel_embeddings))
    if valid_embs:
        sims = cosine_similarity([base_emb], valid_embs)[0]
        for idx, s in zip(valid_indices, sims):
            similarities[idx] = s
            
    top_channels['similarity'] = similarities
    similar_channels = top_channels.sort_values('similarity', ascending=False).head(SIMILAR_CHANNELS_COUNT)
    print(f"Found top {SIMILAR_CHANNELS_COUNT} similar channels.")
    print(similar_channels[['title', 'similarity']].head(10))
else:
    similar_channels = top_channels.head(SIMILAR_CHANNELS_COUNT)
    print("Could not calculate similarities, using top by subscriber count.")

## 5. Fetch Videos and Apply Semantic Threshold

We fetch latest videos from these 100 channels and select the most relevant ones (25-50) based on title similarity to the base channel's content.

In [ ]:
def get_latest_videos(channel_id, max_results=20):
    cache_file = VIDEOS_CACHE_DIR / f"{channel_id}.json"
    if cache_file.exists():
        if (datetime.now().timestamp() - cache_file.stat().st_mtime) < 86400:
            with open(cache_file, 'r') as f:
                return json.load(f)[:max_results]

    if 'youtube' not in globals() or youtube is None: return []
    
    try:
        res_ch = youtube.channels().list(part='contentDetails', id=channel_id).execute()
        if not res_ch['items']: return []
        uploads_playlist_id = res_ch['items'][0]['contentDetails']['relatedPlaylists']['uploads']
        
        # Optimization requested: replace UC with UULF if applicable for long-form
        playlist_id = uploads_playlist_id
        if playlist_id.startswith('UU'):
            playlist_id = 'UULF' + playlist_id[2:]
            
        res_v = youtube.playlistItems().list(
            part='snippet,contentDetails',
            playlistId=playlist_id,
            maxResults=50
        ).execute()
        
        vids = []
        for item in res_v.get('items', []):
            vids.append({
                'video_id': item['contentDetails']['videoId'],
                'title': item['snippet']['title'],
                'channel_id': channel_id,
                'channel_title': item['snippet']['channelTitle']
            })
            
        with open(cache_file, 'w') as f:
            json.dump(vids, f)
        return vids[:max_results]
    except Exception:
        return []

base_videos = get_latest_videos(BASE_CHANNEL_ID, 50)
base_video_titles = [v['title'] for v in base_videos]
base_video_embs = embed_texts(base_video_titles)
base_video_embs = [e for e in base_video_embs if e is not None]
base_avg_emb = np.mean(base_video_embs, axis=0) if base_video_embs else None

all_similar_videos = []
for ch_id in tqdm(similar_channels['id'].tolist(), desc="Fetching similar channel videos"):
    all_similar_videos.extend(get_latest_videos(ch_id, 20))

df_all_vids = pd.DataFrame(all_similar_videos)
if not df_all_vids.empty and base_avg_emb is not None:
    vid_titles = df_all_vids['title'].tolist()
    vid_embs = []
    for i in tqdm(range(0, len(vid_titles), 96), desc="Embedding video titles"):
        vid_embs.extend(embed_texts(vid_titles[i:i+96]))
    
    valid_v_indices = [i for i, e in enumerate(vid_embs) if e is not None]
    valid_v_embs = [vid_embs[i] for i in valid_v_indices]
    
    v_similarities = np.zeros(len(vid_embs))
    if valid_v_embs:
        v_sims = cosine_similarity([base_avg_emb], valid_v_embs)[0]
        for idx, s in zip(valid_v_indices, v_sims):
            v_similarities[idx] = s
            
    df_all_vids['semantic_similarity'] = v_similarities
    
    # Smart threshold: select between 25 and 50
    df_sorted = df_all_vids.sort_values('semantic_similarity', ascending=False)
    if len(df_sorted) > MIN_VIDEOS:
        candidates = df_sorted.iloc[MIN_VIDEOS:MAX_VIDEOS]
        if not candidates.empty:
            sims = candidates['semantic_similarity'].values
            gaps = sims[:-1] - sims[1:]
            if len(gaps) > 0:
                max_gap_idx = np.argmax(gaps)
                split_idx = MIN_VIDEOS + max_gap_idx + 1
                top_vids = df_sorted.head(split_idx)
            else:
                top_vids = df_sorted.head(MAX_VIDEOS)
        else:
            top_vids = df_sorted.head(len(df_sorted))
    else:
        top_vids = df_sorted
    
    video_ids = top_vids['video_id'].tolist()
    stats_map = {}
    if 'youtube' in globals() and youtube is not None:
        for i in range(0, len(video_ids), 50):
            batch = video_ids[i:i+50]
            res = youtube.videos().list(part='statistics', id=','.join(batch)).execute()
            for item in res.get('items', []):
                stats_map[item['id']] = {
                    'view_count': int(item['statistics'].get('viewCount', 0)),
                    'like_count': int(item['statistics'].get('likeCount', 0))
                }
    
    top_vids['view_count'] = top_vids['video_id'].apply(lambda x: stats_map.get(x, {}).get('view_count', 0))
    top_vids['like_count'] = top_vids['video_id'].apply(lambda x: stats_map.get(x, {}).get('like_count', 0))
    
    # Train a new PCA model specific for this channel's competitive universe
    # Using all successfully embedded videos from similar channels for more robust PCA
    all_valid_v_embs = np.array([e for e in vid_embs if e is not None])
    n_components = min(20, len(all_valid_v_embs))
    
    pca = PCA(n_components=n_components, random_state=42)
    pca.fit(all_valid_v_embs)
    
    # Transform only the selected top videos
    valid_top_indices = [i for i in top_vids.index if vid_embs[i] is not None]
    valid_top_embs = np.array([vid_embs[i] for i in valid_top_indices])
    reduced_embs = pca.transform(valid_top_embs)
    
    # Attach to dataframe - note we need to handle potential index mismatches but here indices are stable
    top_vids['embedding_20d'] = list(reduced_embs)
    print(f"✅ Trained and applied new {n_components}D PCA model specific to this channel universe.")
    
    print(f"Selected {len(top_vids)} videos for the final report.")
else:
    top_vids = pd.DataFrame()
    print("No similar videos found or embedding failed.")

## 6. Export Results

Save the competitive universe data and custom PCA model to Google Drive.

In [ ]:
if not top_vids.empty:
    version = datetime.now().strftime('%Y%m%d_%H%M%S')
    sanitized_name = "".join(x for x in BASE_CHANNEL_NAME if x.isalnum())
    export_path = DRIVE_BASE_DIR / 'exports' / 'competitive_universe' / sanitized_name / version
    latest_path = DRIVE_BASE_DIR / 'exports' / 'competitive_universe' / sanitized_name / 'latest'
    
    export_path.mkdir(parents=True, exist_ok=True)
    latest_path.mkdir(parents=True, exist_ok=True)
    
    top_vids.to_csv(export_path / 'competitive_videos.csv', index=False)
    top_vids.to_csv(latest_path / 'competitive_videos.csv', index=False)
    
    # Save the custom PCA model
    if 'pca' in locals():
        joblib.dump(pca, export_path / 'pca_model_20d.joblib')
        joblib.dump(pca, latest_path / 'pca_model_20d.joblib')
    
    print(f"✅ Exported competitive universe and custom PCA model for {BASE_CHANNEL_NAME} to {export_path}")
else:
    print("Nothing to export.")